# Mempools and Oracles — When the Chain Does Not Know Yet

A blockchain can make a shared history hard to rewrite. It cannot learn a payment, a football score, or a market price until somebody sends that information into the network. This notebook starts one step earlier: with the different, incomplete views held by network participants.

**Scope:** these small deterministic models omit signatures, peer selection, fees, validator duties, and live oracle operations. They show causal relationships, not a production protocol.


## Recap

| Notebook | What it established |
| --- | --- |
| 1 | Hash links make an edited history detectable; proof of work makes rewriting costly. |
| 2 | Proof of stake schedules and rewards validators, with stake at risk for bad behaviour. |
| 3 | Merkle trees commit to many records while keeping membership proofs small. |
| 4 | Permissioned chains change who may participate, not the need for shared rules. |

This notebook adds the network layer around those ideas. **Agreement does not create information.** Nodes must first receive a transaction or a block, and that arrival is never perfectly simultaneous.


## 1. One network, many mempools

A mempool is not one global waiting room. It is each node's local list of valid transactions it has heard about so far. Think of it as a group chat in which every phone receives messages in a slightly different order: the messages may be valid, while each local inbox is still incomplete. Two honest nodes can therefore have different mempools at the same moment.

The next definition cell creates those local inboxes and prints nothing; the following gossip demonstration will let us compare their contents.


In [ ]:
import hashlib
import random
import statistics
from dataclasses import dataclass


@dataclass(frozen=True)
class Transaction:
    """One gossiped payment-like fact, identified by its tx_id.

    Attributes:
        tx_id: Unique identifier a node uses to track this transaction
            across its own mempool and others' gossip.
        description: Human-readable summary, for demo printing only.
    """

    tx_id: str
    description: str


class Network:
    """A set of nodes, each holding its own local mempool.

    There is no shared, global mempool here on purpose: `broadcast`
    delivers a transaction to each peer independently and probabilistically,
    modelling how gossip reaches honest nodes at different times.
    """

    def __init__(self, node_names: list[str], rng: random.Random) -> None:
        """Create one empty mempool per node.

        Args:
            node_names: Every node's identity; must be non-empty.
            rng: A seeded ``random.Random`` so a demo run is repeatable.
                This is a classroom convenience, not protocol-grade
                randomness -- real gossip timing is not something a single
                seed could faithfully reproduce anyway, and unlike a
                security-relevant selection (see ``pick_proposer`` below),
                nothing here needs to resist being predicted.

        Raises:
            ValueError: If ``node_names`` is empty.
        """
        if not node_names:
            raise ValueError("A network needs at least one node.")
        self.node_names = list(node_names)
        self.rng = rng
        self.mempools: dict[str, dict[str, Transaction]] = {
            name: {} for name in node_names
        }

    def broadcast(self, transaction: Transaction, origin: str) -> None:
        """Deliver a transaction to its origin, then probabilistically to peers.

        Args:
            transaction: The transaction being gossiped.
            origin: The node that first received/sent it; always gets it.

        Raises:
            ValueError: If ``origin`` is not a known node.
        """
        if origin not in self.mempools:
            raise ValueError(f"Unknown origin node: {origin}")
        self.mempools[origin][transaction.tx_id] = transaction
        for name in self.node_names:
            if name != origin and self.rng.random() < 0.5:
                self.mempools[name][transaction.tx_id] = transaction

    def mempool_ids(self, node_name: str) -> list[str]:
        """Return a node's currently known transaction IDs, sorted.

        Args:
            node_name: The node whose local view to read.

        Returns:
            Sorted list of transaction IDs that node has received so far.

        Raises:
            ValueError: If ``node_name`` is not a known node.
        """
        if node_name not in self.mempools:
            raise ValueError(f"Unknown node: {node_name}")
        return sorted(self.mempools[node_name])

## 2. Gossip: useful precisely because it is imperfect

Gossip sends a transaction outward to many peers. It improves the chance that a future proposer has the transaction, but it does not promise that every peer has it immediately. We use a seeded random generator so this small demonstration is repeatable; that classroom seed is **not** protocol-grade randomness. Watch for different transaction lists at honest nodes and the average local size, rather than a claim that one node holds the network's definitive mempool.

> Pause and predict: after four broadcasts, will every node have all four transaction IDs?


In [1]:
nodes = ["Node A", "SketchyGuy-Node", "Emma-Node", "Farid-Node"]
network = Network(nodes, random.Random(7))
transactions = [
    Transaction("tx1", "Alice pays Bob"),
    Transaction("tx2", "Carol pays Titus"),
    Transaction("tx3", "Titus pays Alice"),
    Transaction("tx4", "Bob pays Carol"),
]
origins = ["Node A", "SketchyGuy-Node", "Emma-Node", "Node A"]

for transaction, origin in zip(transactions, origins):
    network.broadcast(transaction, origin)

print("Each node has its own mempool:")
for node in nodes:
    print(f"  {node}: {network.mempool_ids(node)}")

mempool_sizes = [len(network.mempool_ids(node)) for node in nodes]
print(f"Average local mempool size: {statistics.mean(mempool_sizes):.2f}")


Each node has its own mempool:
  Node A: ['tx1', 'tx2', 'tx3', 'tx4']
  SketchyGuy-Node: ['tx1', 'tx2', 'tx4']
  Emma-Node: ['tx1', 'tx3', 'tx4']
  Farid-Node: ['tx2', 'tx3', 'tx4']
Average local mempool size: 3.25


**Read the result:** Node A has all four IDs, while every other honest node is missing one. The four states are deliberately different: **origin** means the sender gave a node the transaction; **gossip receipt** means another node heard it; **inclusion** means a proposer put it in a block; and **confirmation** means the network later treats that block as part of its accepted history. A transaction can be at any earlier state without reaching the later ones.


## 3. From transactions to block proposals

A proposer can only choose from its local view. In a real chain it also checks signatures, fees, execution rules, and block limits; this notebook keeps only transaction IDs so we can see how incomplete propagation affects the next block. A PoS schedule may be known, but directly sending a transaction to the scheduled proposer is still brittle and censorable: that one node might be delayed, unavailable, or unwilling to relay it. Direct delivery is also insufficient for network-wide validation and failover, so gossip remains the way many independently validating peers can receive the transaction.


## 4. Slots: one scheduled proposer, no mining race

Proof of stake divides time into slots and assigns one eligible proposer to each slot. The assignment must be reproducible by all validators, so this model hashes a documented epoch seed and slot number. It is a teaching stand-in for a real protocol's randomness and eligibility proofs. The next definition cell prints nothing; it imports `Validator`, `Block`, `Blockchain`, `pick_proposer`, and `slash` from `blockchain_lib.pos` -- the same module notebook 2 Part 1 is built from, kept in one place instead of redeclared per notebook -- and adds the slot-based `assign_proposer` needed to show how a delayed block can produce two honest proposals. `slash` and `pick_proposer` are imported for completeness but never called here -- this notebook's fork has nobody to punish.


In [ ]:
import sys
from pathlib import Path


def _repo_root(marker: str = "pyproject.toml") -> Path:
    """Walk upward from the current working directory to find the repo root."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"Could not find {marker} above {Path.cwd()}")


_ROOT = _repo_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from blockchain_lib.pos import Block, Blockchain, Validator


def assign_proposer(
    validators: list[Validator], slot: int, epoch_seed: str
) -> Validator:
    """Deterministically pick a slot's proposer from a stake-weighted lottery.

    Unlike ``pick_proposer`` (OS CSPRNG, unpredictable by design), this
    hashes a documented ``epoch_seed`` and ``slot`` number so every
    validator can reproduce the same assignment independently -- a real
    protocol needs that reproducibility (a stand-in for VRF/RANDAO-style
    eligibility proofs), whereas per-call proposer selection does not.

    Args:
        validators: Candidate validators; must have positive total stake.
        slot: The slot number being assigned.
        epoch_seed: A documented, shared seed for this epoch.

    Returns:
        The ``Validator`` assigned to this slot.

    Raises:
        ValueError: If there are no validators or total stake is not positive.
    """
    if not validators or sum(v.stake for v in validators) <= 0:
        raise ValueError("Proposer assignment needs positive validator stake.")
    seed = hashlib.sha256(f"{epoch_seed}:{slot}".encode()).digest()
    pick = int.from_bytes(seed, "big") % sum(v.stake for v in validators)
    cumulative = 0
    for validator in validators:
        cumulative += validator.stake
        if pick < cumulative:
            return validator
    raise RuntimeError("Unreachable proposer-selection state.")

## 5. A fork caused by delay, not dishonesty

We use the stable epoch seed `notebook-5`. With the four validators below it assigns Farid-Node to slot 1 and Node A to slot 2. They are different scheduled proposers. The slot 2 proposer has not received Block #1, so it honestly builds on the latest tip it knows: Genesis. Both candidate blocks therefore point to Genesis. The output will name both proposers and draw the two arrows leaving Genesis.

> Pause and predict: if the next proposer has not seen a valid earlier block, can deterministic proposer assignment alone prevent a fork?


In [2]:
validators = [
    Validator("Node A", 100),
    Validator("SketchyGuy-Node", 80),
    Validator("Emma-Node", 70),
    Validator("Farid-Node", 50),
]
validators_by_name = {validator.name: validator for validator in validators}
epoch_seed = "notebook-5"

chain = Blockchain(validators)
genesis = chain.chain[0]
slot_1_proposer = assign_proposer(validators, 1, epoch_seed)
slot_2_proposer = assign_proposer(validators, 2, epoch_seed)
assert slot_1_proposer.name != slot_2_proposer.name

block_1 = chain.propose_candidate(
    ", ".join(network.mempool_ids(slot_1_proposer.name)), slot_1_proposer.name
)
proposal_time_views = {
    "Node A": {genesis.hash},
    "SketchyGuy-Node": {genesis.hash, block_1.hash},
    "Emma-Node": {genesis.hash, block_1.hash},
    "Farid-Node": {genesis.hash, block_1.hash},
}
assert block_1.hash not in proposal_time_views[slot_2_proposer.name]
block_2_alt = chain.propose_candidate(
    ", ".join(network.mempool_ids(slot_2_proposer.name)), slot_2_proposer.name
)

print(f"Slot 1 proposer: {slot_1_proposer.name} creates Block #1.")
print(
    f"Slot 2 proposer: {slot_2_proposer.name} has NOT received Block #1; "
    "it builds on Genesis."
)
print("\nPropagation fork (both arrows leave Genesis):")
print("          Genesis")
print("     +----+----+")
print("     |         |")
print("   Block #1    Block #2-alt")
print(f"   {slot_1_proposer.name:<12} {slot_2_proposer.name}")

Slot 1 proposer: Farid-Node creates Block #1.
Slot 2 proposer: Node A has NOT received Block #1; it builds on Genesis.

Propagation fork (both arrows leave Genesis):
          Genesis
     +----+----+
     |         |
   Block #1    Block #2-alt
   Farid-Node   Node A


**Read the result:** Farid-Node proposes Block #1, while Node A honestly proposes Block #2-alt from Genesis because its local view is delayed. A fork is therefore a temporary disagreement about the latest valid tip, not evidence that either proposer cheated.


**Aside — is this equivocation?** Notebook 2 slashes validators for provable double-signing: one identity, two conflicting blocks, the same height. This fork is not that. Farid-Node and Node A are two *different* validators, each honestly proposing for their own correctly assigned slot. `detect_equivocation` and `slash` are imported from `blockchain_lib.pos` and available here; the next cell uses `detect_equivocation` to check this formally, instead of just asserting in prose that nobody did anything wrong.

> Pause and predict: will `chain.detect_equivocation` treat Block #1 and Block #2-alt as two conflicting proposals from Farid-Node?


In [3]:
print("Is this fork equivocation? Check formally instead of asserting it:")
try:
    chain.detect_equivocation(1, slot_1_proposer.name, [block_1, block_2_alt])
except ValueError as error:
    print(f"  detect_equivocation refuses to compare them: {error}")

print(
    "\nBlock #1 and Block #2-alt were proposed by two different validators "
    f"({slot_1_proposer.name} and {slot_2_proposer.name}), each correctly "
    "assigned to their own slot -- not one validator signing twice for the "
    "same slot. There is no equivocation here, so slash() has nothing to "
    "punish."
)

Is this fork equivocation? Check formally instead of asserting it:
  detect_equivocation refuses to compare them: detect_equivocation compares one proposer's own candidates; got block(s) proposed by ['Node A'], not Farid-Node. Different proposers disagreeing is a fork, not equivocation.

Block #1 and Block #2-alt were proposed by two different validators (Farid-Node and Node A), each correctly assigned to their own slot -- not one validator signing twice for the same slot. There is no equivocation here, so slash() has nothing to punish.


**Read the result:** `detect_equivocation` refuses to even compare Block #1 and Block #2-alt as Farid-Node's candidates, because Block #2-alt was never proposed by Farid-Node -- it is Node A's block. Equivocation requires one identity signing twice for the same height; two different validators disagreeing is an ordinary fork. `slash` is reused and available, but nothing here calls it.


## 6. Attestations turn local views into fork-choice weight

After propagation continues, validators attest to the candidate tip they have received. These votes are explicit observations, not random outcomes. Our tiny fork-choice model adds the stake behind each candidate and deliberately rejects ties, because a real protocol needs a specified tie-break rule too. Watch for the votes and their 200-versus-100 stake weights: fork choice weighs received views; it does not repair the earlier delay.

In [4]:
def attestation_weight(
    tip_hash: str,
    votes: dict[str, str],
    validators_by_name: dict[str, Validator],
) -> int:
    """Sum the stake of every validator whose vote names ``tip_hash``.

    Args:
        tip_hash: Candidate block hash being scored.
        votes: Mapping of validator name to the block hash they voted for.
        validators_by_name: All known validators, keyed by name.

    Returns:
        Total stake behind ``tip_hash``.

    Raises:
        ValueError: If a vote names an attester outside ``validators_by_name``.
    """
    unknown = set(votes) - set(validators_by_name)
    if unknown:
        raise ValueError(f"Unknown attesters: {sorted(unknown)}")
    return sum(
        validators_by_name[name].stake
        for name, voted_hash in votes.items()
        if voted_hash == tip_hash
    )


def fork_choice_two_tips(
    candidate_tips: list[Block],
    votes: dict[str, str],
    validators_by_name: dict[str, Validator],
) -> tuple[Block, dict[str, int]]:
    """Pick the candidate tip with the most attestation-weighted stake.

    Args:
        candidate_tips: The rival blocks to choose between.
        votes: Mapping of validator name to the block hash they voted for.
        validators_by_name: All known validators, keyed by name.

    Returns:
        ``(winning_block, weights)`` where ``weights`` maps each
        candidate's hash to its total attested stake.

    Raises:
        ValueError: If a vote names a hash outside ``candidate_tips``, or
            if two candidates tie (this teaching model has no tie-break
            rule -- a real protocol needs one).
    """
    candidate_hashes = {block.hash for block in candidate_tips}
    invalid_votes = set(votes.values()) - candidate_hashes
    if invalid_votes:
        raise ValueError("An attestation names an unknown candidate tip.")
    weights = {
        block.hash: attestation_weight(block.hash, votes, validators_by_name)
        for block in candidate_tips
    }
    if len(set(weights.values())) != len(weights):
        raise ValueError("Teaching model requires an explicit tie-break rule.")
    winner_hash = max(weights, key=weights.get)
    return next(block for block in candidate_tips if block.hash == winner_hash), weights


# By attestation time, Block #1 (slot 1) has had longer to propagate, so every
# node that already had it in `proposal_time_views` still has it. Block #2-alt
# is brand new: only its own proposer has received it so far. No node holds
# both, so each vote below is just "the block this node has actually seen" —
# read out of `proposal_time_views`, not hand-picked to fit a story.
attestation_time_views = dict(proposal_time_views)
attestation_time_views[slot_2_proposer.name] = attestation_time_views[
    slot_2_proposer.name
] | {block_2_alt.hash}


def vote_for_known_tip(known_block_hashes: set[str]) -> str:
    """Return the block hash a validator with this local view would vote for.

    Prefers Block #1 whenever it is known -- in this scenario no validator
    ever holds both candidates at once, so this never needs a real
    tie-break rule between two simultaneously-known tips.
    """
    return block_1.hash if block_1.hash in known_block_hashes else block_2_alt.hash


votes = {
    name: vote_for_known_tip(known_hashes)
    for name, known_hashes in attestation_time_views.items()
}

print("\nView-derived attestations:")
for name, voted_hash in votes.items():
    label = "Block #1" if voted_hash == block_1.hash else "Block #2-alt"
    print(f"  {name} received {label} and votes for {label}.")

winner, weights = fork_choice_two_tips(
    [block_1, block_2_alt], votes, validators_by_name
)
winner_label = "Block #1" if winner.hash == block_1.hash else "Block #2-alt"
print(f"Block #1 stake weight: {weights[block_1.hash]}")
print(f"Block #2-alt stake weight: {weights[block_2_alt.hash]}")
print(f"Canonical tip: {winner_label}")

chain.accept_candidate(winner)
valid, message = chain.is_valid()
print(
    f"Chain length after fork resolution: {len(chain.chain)}; "
    f"valid? {valid} -- {message}"
)


View-derived attestations:
  Node A received Block #2-alt and votes for Block #2-alt.
  SketchyGuy-Node received Block #1 and votes for Block #1.
  Emma-Node received Block #1 and votes for Block #1.
  Farid-Node received Block #1 and votes for Block #1.
Block #1 stake weight: 200
Block #2-alt stake weight: 100
Canonical tip: Block #1
Chain length after fork resolution: 2; valid? True -- Chain is valid.


**Read the result:** Block #1 wins because SketchyGuy-Node, Emma-Node, and Farid-Node already had it in their local view, contributing 80 + 70 + 50 = 200 stake, versus 100 stake from Node A, which has only its own Block #2-alt proposal. Fork choice follows the attestation weight behind each received view, not the block's slot number.

## 7. PoW and PoS: keep three questions separate

| Question | Proof of work | Proof of stake in this model |
| --- | --- | --- |
| Who may propose? | Any miner that wins the hash puzzle race. | One scheduled proposer for the slot. |
| Is this block valid? | Every node checks the block against shared consensus rules, including its proof of work. | Every node deterministically checks the block against consensus state and the scheduled proposer's signature; the shared seed helps verify eligibility. |
| Why can a fork appear? | Competing miners can find blocks near the same time. | A scheduled proposer can still be missing a recently propagated block. |
| How is a tip chosen? | More cumulative proof of work. | More stake-weighted attestations among valid branches. |

The lesson is not that PoS eliminates networking. It changes the permission to propose and the fork-choice evidence, while ordinary delay can still give honest participants different local views. The next part of this notebook uses the same boundary between off-chain facts and on-chain agreement to study oracles.


# Part 2 — Information from outside the chain

The fork lesson showed how a network agrees once it has received information. Now we ask a different question: who gives a contract information that does not already live on the chain?


## 8. Oracles: the chain cannot look outside itself

A smart contract can deterministically inspect its own state and transaction inputs. It cannot directly check an exchange screen, a weather station, or an ETH/USD market. An **oracle** is the trust boundary: someone or something reports an outside fact in a form the contract can use. Agreement about a report does not prove that the outside fact was true. We need it now because the next examples ask a lending rule to value ETH in dollars. The following definition cell prints nothing; it supplies the report and collateral-ratio helpers whose outputs will make that trust boundary visible.

> Pause and predict: if a lending contract trusts one reported ETH price, what can happen when that report is wrong?


In [ ]:
@dataclass(frozen=True)
class PriceReport:
    """One reported ETH/USD price from a named source.

    Attributes:
        source: Who or what reported this price (for demo printing/trust
            reasoning only -- nothing here authenticates the source).
        eth_price_usd: The reported price.
    """

    source: str
    eth_price_usd: float


def collateral_ratio(
    collateral_eth: float, debt_usd: float, eth_price_usd: float
) -> float:
    """Return collateral value divided by debt, at a given ETH price.

    Args:
        collateral_eth: ETH locked as collateral. Must be positive.
        debt_usd: Outstanding debt in USD. Must be positive.
        eth_price_usd: ETH price used to value the collateral. Must be
            positive.

    Returns:
        ``collateral_eth * eth_price_usd / debt_usd``.

    Raises:
        ValueError: If any argument is not positive.
    """
    if collateral_eth <= 0 or debt_usd <= 0 or eth_price_usd <= 0:
        raise ValueError("Collateral, debt, and price must be positive.")
    return collateral_eth * eth_price_usd / debt_usd


def should_liquidate(ratio: float, threshold: float = 1.5) -> bool:
    """Return whether a collateral ratio is below the liquidation threshold."""
    return ratio < threshold


def median_reported_price(reports: list[PriceReport]) -> float:
    """Return the median ETH price across several independent reports.

    Args:
        reports: Non-empty list of price reports to aggregate.

    Returns:
        The median ``eth_price_usd`` across ``reports``.

    Raises:
        ValueError: If ``reports`` is empty.
    """
    if not reports:
        raise ValueError("At least one price report is required.")
    return statistics.median(report.eth_price_usd for report in reports)

## 9. One report is a single point of failure

A contract follows the value it receives, even if the source is malicious. The next code cell prices 10 ETH at a false $1,200 and prints the resulting collateral ratio and liquidation verdict. Watch for correct arithmetic producing the wrong business outcome.


In [5]:
collateral_eth = 10
debt_usd = 12_000
bad_report = PriceReport("malicious-feed", 1_200)
bad_report_ratio = collateral_ratio(
    collateral_eth, debt_usd, bad_report.eth_price_usd
)

print(f"Position: {collateral_eth} ETH collateral / ${debt_usd:,} debt")
print(f"Single report: ${bad_report.eth_price_usd:,.0f}/ETH")
print(f"Single-source collateral ratio: {bad_report_ratio:.2f}")
if should_liquidate(bad_report_ratio):
    print("Single-source verdict: LIQUIDATE (wrong)")


Position: 10 ETH collateral / $12,000 debt
Single report: $1,200/ETH
Single-source collateral ratio: 1.00
Single-source verdict: LIQUIDATE (wrong)


**Read the result:** the false $1,200 price values 10 ETH at $12,000, exactly matching the $12,000 debt, so the collateral ratio is 1.00. Because 1.00 is below the 1.5 threshold, the contract correctly applies its rule but reaches the wrong verdict: it would liquidate a healthy position because its single input was false.


## 10. Aggregation raises the cost of lying

One simple defence is to combine several reports and use their median. Here two nearby reports surround one malicious outlier. This only helps when the sources are independent and fresh; median aggregation alone is not a production oracle design. The output will show the middle report and whether its ratio clears the threshold.

> Pause and predict: which of $2,005, $1,200, and $1,995 becomes the median price?


In [6]:
reports = [
    PriceReport("independent-feed-a", 2_005),
    PriceReport("malicious-feed", 1_200),
    PriceReport("independent-feed-b", 1_995),
]
median_price = median_reported_price(reports)
median_report_ratio = collateral_ratio(collateral_eth, debt_usd, median_price)

print("Reports: $2,005, $1,200, $1,995 per ETH")
print(f"Median report: ${median_price:,.0f}/ETH")
print(f"Median collateral ratio: {median_report_ratio:.2f}")
if not should_liquidate(median_report_ratio):
    print("Median verdict: HEALTHY")


Reports: $2,005, $1,200, $1,995 per ETH
Median report: $1,995/ETH
Median collateral ratio: 1.66
Median verdict: HEALTHY


**Read the result:** the median ignores the lone $1,200 outlier and keeps the $1,995 report. The ratio is 1.66, above the 1.5 threshold, so this position remains healthy. Aggregation raises the cost of lying; it does not remove the need to examine source independence, freshness, and incentives.


## Next: one transaction, one temporary price, one very bad day

What happens when a protocol trusts a manipulable AMM spot price inside one transaction? Continue with [6. oracle_manipulation_and_flash_loans.ipynb](6.%20oracle_manipulation_and_flash_loans.ipynb).


## Takeaways

- **Mechanism:** each node keeps a local mempool and block view. **Not a guarantee:** honest nodes do not receive information at the same instant.
- **Mechanism:** PoS fork choice weighs attestations. **Not a guarantee:** it cannot make information arrive instantly or make every temporary fork impossible.
- **Mechanism:** an oracle introduces an outside fact for a contract to use. **Not a guarantee:** the supplied fact is automatically true.
- **Mechanism:** several independent, fresh reports can resist one outlier. **Not a guarantee:** aggregation magically proves reality.
